In [1]:
# ============================================================
# FLOOD DETECTION v5.1 — REPAIR & OPTIMIZE
# Fixed: NameError 'smp', Device Placement, and RLE Logic
# ============================================================

import os, sys, subprocess

# 1. Force Install & Import
def install_and_import():
    try:
        import segmentation_models_pytorch as smp
    except ImportError:
        print("Installing segmentation-models-pytorch...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "segmentation-models-pytorch"])
        import segmentation_models_pytorch as smp
    return smp

smp = install_and_import()

import random, warnings, rasterio, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import albumentations as A
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ── Configuration ─────────────────────────────────────────────────────────────
BASE      = Path('/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data')
IMAGE_DIR = BASE / 'image'
LABEL_DIR = BASE / 'label'
PRED_DIR  = BASE / 'prediction/image'
SPLIT_DIR = BASE / 'split'

NUM_CLASSES  = 3 
NUM_CHANNELS = 11 
PATCH_SIZE   = 512
BATCH_SIZE   = 8 
EPOCHS       = 30
LR           = 3e-4
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================
# STEP 1: Feature Engineering
# ============================================================
def get_advanced_features(img):
    vv, vh = img[0], img[1]
    ndwi = ((img[2] - img[4]) / (img[2] + img[4] + 1e-8))[np.newaxis]
    mndwi = ((img[2] - img[5]) / (img[2] + img[5] + 1e-8))[np.newaxis]
    ndvi = ((img[4] - img[3]) / (img[4] + img[3] + 1e-8))[np.newaxis]
    ndmi = ((img[4] - img[5]) / (img[4] + img[5] + 1e-8))[np.newaxis]
    sar_ratio = (vv / (vh + 1e-8))[np.newaxis]

    feats = np.concatenate([ndwi, mndwi, ndvi, ndmi, sar_ratio], axis=0)
    return np.concatenate([img, np.clip(feats, -1, 1)], axis=0)

# ============================================================
# STEP 2: Dataset
# ============================================================
class ProFloodDataset(Dataset):
    def __init__(self, ids, image_dir, label_dir=None, augment=False):
        self.ids = ids
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir) if label_dir else None
        self.augment = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Transpose(p=0.5),
        ]) if augment else None

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        sid = self.ids[idx]
        with rasterio.open(self.image_dir / f'{sid}_image.tif') as src:
            img = get_advanced_features(src.read().astype(np.float32))
        
        for i in range(NUM_CHANNELS):
            img[i] = (img[i] - img[i].mean()) / (img[i].std() + 1e-8)

        if self.label_dir:
            with rasterio.open(self.label_dir / f'{sid}_label.tif') as src:
                lbl = src.read(1).astype(np.int64)
            lbl = np.clip(np.nan_to_num(lbl, nan=0), 0, NUM_CLASSES-1)
            if self.augment:
                aug = self.augment(image=img.transpose(1,2,0), mask=lbl)
                img, lbl = aug['image'].transpose(2,0,1), aug['mask']
            return torch.from_numpy(img).float(), torch.from_numpy(lbl).long()
        return torch.from_numpy(img).float(), sid

# ============================================================
# STEP 3: Model & Loss
# ============================================================
def build_pro_model():
    # EfficientNet-B5 usually provides a better balance for 0.15+ scores
    model = smp.DeepLabV3Plus(
        encoder_name="efficientnet-b5", 
        encoder_weights="imagenet",
        in_channels=NUM_CHANNELS, 
        classes=NUM_CLASSES
    ).to(DEVICE)
    return model

class ProLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode='multiclass')
        self.tversky = smp.losses.TverskyLoss(mode='multiclass', alpha=0.7, beta=0.3)

    def forward(self, y_pr, y_gt):
        return 0.5 * self.dice(y_pr, y_gt) + 0.5 * self.tversky(y_pr, y_gt)

# ============================================================
# STEP 4: Training Loop
# ============================================================
def rle_encode(mask):
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    if len(runs) % 2 != 0: runs = np.append(runs, len(pixels) - 1)
    res = np.zeros(len(runs), dtype=int)
    res[0::2], res[1::2] = runs[::2], runs[1::2] - runs[::2]
    return ' '.join(str(x) for x in res)

def train_pro(model, loader):
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = ProLoss()
    scaler = GradScaler()

    for epoch in range(EPOCHS):
        model.train()
        for imgs, masks in tqdm(loader, desc=f'Epoch {epoch+1}'):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                loss = criterion(model(imgs), masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
    return model

# ============================================================
# STEP 5: Execution
# ============================================================
if __name__ == "__main__":
    def load_ids(f): return [l.strip() for l in open(SPLIT_DIR / f) if l.strip()]
    train_ids, pred_ids = load_ids('train.txt'), load_ids('pred.txt')

    train_loader = DataLoader(ProFloodDataset(train_ids, IMAGE_DIR, LABEL_DIR, True), 
                              batch_size=BATCH_SIZE, shuffle=True)
    pred_loader = DataLoader(ProFloodDataset(pred_ids, PRED_DIR), batch_size=1)

    model = build_pro_model()
    model = train_pro(model, train_loader)

    model.eval()
    results = []
    with torch.no_grad():
        for imgs, sids in tqdm(pred_loader, desc='Inference'):
            imgs = imgs.to(DEVICE)
            with autocast():
                # TTA: Vertical Flip + Original
                out1 = torch.softmax(model(imgs), dim=1)
                out2 = torch.flip(torch.softmax(model(torch.flip(imgs, [2])), dim=1), [2])
                final_pred = (out1 + out2).argmax(dim=1).cpu().numpy()
            
            for i in range(len(sids)):
                results.append({'id': sids[i], 'rle_mask': rle_encode((final_pred[i] == 1).astype(np.uint8))})

    pd.DataFrame(results).to_csv('submission.csv', index=False)
    print("🎉 Done. Check submission.csv")

Installing segmentation-models-pytorch...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.3 MB/s eta 0:00:00


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

Inference: 100%|██████████| 19/19 [00:10<00:00,  1.86it/s]

🎉 Done. Check submission.csv
